## Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics and debugging.
- Transforming prompts, tool selection, and output formatiing.
- Adding retries, fallbacks and early termination logic.
- Applying rate limits, guardrails and PII detection.

In [1]:
pip install -U langchain-google-genai langchain -quiet

Note: you may need to restart the kernel to use updated packages.


c:\Users\chsha\Desktop\KrishNaik Course\Lanchain_Updated\.venv\Scripts\python.exe: No module named pip


In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Set your API key (get this from Google AI Studio)
os.environ["GEMINI_API_KEY"] = 

# Initialize the model (using gemini-3.5-flash, the engine behind Antigravity)
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.7,
    max_retries=2
)


## Summarization MiddleWare

Automatically summarizes conversation history when apprcahing token limits, preserving recent messages while compressing older context

Summratization is useful for the following:
- Long-running conversations that exveed context windows.
- Multi-turn dialogyes with extensive history.
- Applciations where preserving full conversation context matters.

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent=create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [ ]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [ ]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

## Token Size

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [5]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~142 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='b7da8306-2288-4cc8-ac21-987c47eae4e5'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'abc0cb65-6b29-4c06-a48d-b2e0cc93a050': 'EjQKMgEMOdbH+HfBlsxka6x0fscxmueC7tWNtui0wzlsYcO7johKsQlT3X8jGWCy4KGUeR0F'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e53e0-8fa4-7f52-adf4-0335463217c2-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'abc0cb65-6b29-4c06-a48d-b2e0cc93a050', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens': 16, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Paris:\n    1. Grand Hotel - 5 star, $350/night,

## Fraction


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

## Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [7]:
agent=create_agent(
    model=llm,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [8]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [9]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='4057da8d-a853-4c14-bf74-0f0439b7f3a6'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'f6590010-0b2f-4a08-8e2c-4ee6b4be36b7': 'EjQKMgEMOdbHoZVd2x2ZvckpmrLMSXZZmLAQrzeEdOdXbhnV7bVdLaRpwnjRtrutYOKviDL/'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e53e6-b68b-7f63-95d0-ddf6ec197cab-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'recipient': 'john@test.com', 'body': 'How are you?'}, 'id': 'f6590010-0b2f-4a08-8e2c-4ee6b4be36b7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'output_t

In [10]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': "OK. I've sent the email to john@test.com.", 'extras': {'signature': 'EjQKMgEMOdbHZ3yZlKU2L5eEj83ZWkye5S8vUpiEi3TsceUVxsmOWImea8N6e6Xit49Xkw8t'}}]


In [11]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='4057da8d-a853-4c14-bf74-0f0439b7f3a6'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'f6590010-0b2f-4a08-8e2c-4ee6b4be36b7': 'EjQKMgEMOdbHoZVd2x2ZvckpmrLMSXZZmLAQrzeEdOdXbhnV7bVdLaRpwnjRtrutYOKviDL/'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e53e6-b68b-7f63-95d0-ddf6ec197cab-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'recipient': 'john@test.com', 'body': 'How are you?'}, 'id': 'f6590010-0b2f-4a08-8e2c-4ee6b4be36b7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'output_t